# Comparable Company Analysis

**Purpose:** Demonstrate a compact peer-based relative-value workflow using the `finstack.statements_analytics` comparable-company helpers.

**Prerequisites:** Familiarity with common credit and valuation ratios such as EV/EBITDA, leverage, interest coverage, and spread.

**In this notebook:** We place a subject issuer inside a peer set, compute ranks and z-scores, fit a spread-vs-leverage line, and combine several dimensions into one rich/cheap signal.


## Concept

A useful peer workflow usually answers three questions:

1. Where does the subject sit in the peer distribution?
2. What spread looks fair given one key driver, such as leverage?
3. Do multiple dimensions agree that the issuer looks rich or cheap?


In [ ]:
from finstack.statements_analytics import (
    compute_multiple,
    peer_stats,
    percentile_rank,
    regression_fair_value,
    score_relative_value,
    z_score,
)


def banner(title: str) -> None:
    print(f"\n{'=' * 72}")
    print(title)
    print('=' * 72)


subject = {
    "ev_ebitda": 8.5,
    "leverage": 3.5,
    "interest_coverage": 4.2,
    "oas_bps": 350.0,
}

peers = [
    {"ev_ebitda": 7.2, "leverage": 2.5, "interest_coverage": 5.5, "oas_bps": 250.0},
    {"ev_ebitda": 8.0, "leverage": 3.0, "interest_coverage": 4.8, "oas_bps": 300.0},
    {"ev_ebitda": 8.8, "leverage": 3.8, "interest_coverage": 4.0, "oas_bps": 380.0},
    {"ev_ebitda": 9.5, "leverage": 4.5, "interest_coverage": 3.3, "oas_bps": 450.0},
    {"ev_ebitda": 7.8, "leverage": 2.8, "interest_coverage": 5.0, "oas_bps": 280.0},
    {"ev_ebitda": 8.3, "leverage": 3.3, "interest_coverage": 4.5, "oas_bps": 330.0},
    {"ev_ebitda": 9.0, "leverage": 4.0, "interest_coverage": 3.8, "oas_bps": 400.0},
    {"ev_ebitda": 10.2, "leverage": 5.0, "interest_coverage": 2.8, "oas_bps": 520.0},
    {"ev_ebitda": 7.5, "leverage": 2.2, "interest_coverage": 6.0, "oas_bps": 220.0},
    {"ev_ebitda": 8.6, "leverage": 3.6, "interest_coverage": 4.1, "oas_bps": 360.0},
]

metrics = ("ev_ebitda", "leverage", "interest_coverage", "oas_bps")
peer_series = {metric: [peer[metric] for peer in peers] for metric in metrics}


## Peer positioning and fair value

The next cell starts with a subject issuer, compares it against a ten-name peer set, and then blends the results into a composite relative-value score. This is a good notebook-scale example of how the analytics module can support a credit memo or screening process.


In [ ]:
banner("Subject company metrics")
for key, value in subject.items():
    print(f"{key:<20}: {value:>8.2f}")

subject_company = {"enterprise_value": 8_500.0, "ebitda": 1_000.0}
print(
    f"\ncompute_multiple(subject_company, 'EvEbitda') = "
    f"{compute_multiple(subject_company, 'EvEbitda'):.2f}x"
)

banner("Subject vs peers")
print(f"{'metric':<20} {'subject':>10} {'pctile':>10} {'z-score':>10}")
for metric in metrics:
    pctile = percentile_rank(subject[metric], peer_series[metric])
    z = z_score(subject[metric], peer_series[metric])
    print(f"{metric:<20} {subject[metric]:>10.2f} {pctile:>9.1f}% {z:>+10.2f}")

banner("Regression fair value")
reg = regression_fair_value(
    peer_series["leverage"],
    peer_series["oas_bps"],
    subject["leverage"],
    subject["oas_bps"],
)
print(f"slope              : {reg['slope']:.2f} bps / turn")
print(f"intercept          : {reg['intercept']:.2f} bps")
print(f"R-squared          : {reg['r_squared']:.3f}")
print(f"fitted spread      : {reg['fitted_value']:.1f} bps")
print(f"actual spread      : {subject['oas_bps']:.1f} bps")
print(f"residual           : {reg['residual']:+.1f} bps")

banner("Peer distribution summary")
for metric in metrics:
    stats = peer_stats(peer_series[metric])
    print(
        f"{metric:<20} n={stats['n']:>2} median={stats['median']:.2f} "
        f"mean={stats['mean']:.2f} std={stats['std_dev']:.2f}"
    )

banner("Composite score")
dimensions = [
    ("oas_bps", 0.40),
    ("ev_ebitda", 0.30),
    ("leverage", 0.15),
    ("interest_coverage", 0.15),
]
score = score_relative_value(subject, peers, dimensions)
print(f"composite_score : {score['composite_score']:+.3f}")
print(f"confidence      : {score['confidence']:.3f}")
print(f"peer_count      : {score['peer_count']}")
for name, _ in dimensions:
    dim = score['by_dimension'].get(name)
    if dim is None:
        continue
    print(
        f"  {name:<20} pctile={dim['percentile'] * 100:>6.1f}% "
        f"z={dim['z_score']:>+7.2f} weight={dim['weight']:.2f}"
    )


## Takeaways

- `percentile_rank()` and `z_score()` answer slightly different questions and are both useful.
- `regression_fair_value()` is a compact way to anchor relative value on one driver.
- `score_relative_value()` lets you combine several noisy dimensions into one transparent signal instead of relying on one ratio alone.


In [ ]:
{
    "fair_spread_bps": round(reg['fitted_value'], 1),
    "actual_spread_bps": round(subject['oas_bps'], 1),
    "spread_residual_bps": round(reg['residual'], 1),
    "composite_score": round(score['composite_score'], 3),
}
